# Giai đoạn 2 trên Kaggle (2 × T4)

Chạy các cấu hình trong `CONFIGS` với các seed trong `SEEDS`; các lần train được chia đều lên 2 GPU, rồi đánh giá mọi checkpoint.

Cấu hình: `ppo` = PPO gốc với 14 reward viết tay (mốc chuẩn); `v7c` = GAIL; `bc` = GAIL + behavior cloning; `bctask` = behavior cloning + chỉ task reward (không GAIL); `sched`, `schedbc` = biến thể có lịch trình λ.

**Cải tiến GAIL + BC** (mặc định hiện tại): `bcr` = `bc` + phạt độ giật hành động (term viết tay thứ 3); `bcrs` = `bcr` + độ lệch chuẩn hành động ban đầu 0,3; `bcrsw` = `bcrs` + 25 iteration đầu chỉ cập nhật critic. `bcrs1`, `bcrs2` = `bcrs` với mức phạt độ giật nhẹ hơn (−0,03 và −0,06 thay vì −0,1), mặc định hiện tại. Checkpoint được lưu và đánh giá mỗi `EVAL_EVERY` iteration; bảng kết quả in cho cả ngưỡng 90% và 95%.

**Thí nghiệm tái sử dụng** (`VARIANT` = `payload`, `weak` hoặc `slippery`): train trên robot đã thay đổi (thêm 6 kg ở thân, PD gain còn 70%, ma sát chân thấp) nhưng dữ liệu mẫu vẫn lấy từ expert của robot gốc. Mốc hội tụ lấy theo PPO gốc đã hội tụ trên chính biến thể đó, nên `CONFIGS` phải có `ppo` (ví dụ `ppo bc`). Để trống `VARIANT` để chạy robot gốc như trước.

**Trước khi chạy:**
1. *Settings → Accelerator:* **GPU T4 x2**.
2. *Settings → Internet:* **On**.
3. Chỉnh ô tiếp theo nếu cần (mặc định: robot gốc, cấu hình `bcrs1 bcrs2`, seed 1 2 3, 750 iteration, đánh giá mỗi 50 iteration).
4. Bấm **Save Version → Save & Run All (Commit)** để chạy nền (tối đa 12 giờ).

Kết quả: file `variant-<tên>_results-*.tgz` (hoặc `phase2_results-*.tgz`) trong tab **Output**.

In [ ]:
VARIANT = ""                       # "" = robot gốc; "weak", "slippery", "payload" = thí nghiệm tái sử dụng (cần "ppo" trong CONFIGS)
CONFIGS = "bcrs1 bcrs2"             # train: "ppo", "bc", "bcr", "bcrs", "bcrs1", "bcrs2", "bcrsw", "bctask", "v7c", "sched", "schedbc"
SEEDS = "1 2 3"                    # các seed
ITERS = 750                        # số iteration mỗi lần train
EVAL_EVERY = 50                    # lưu và đánh giá checkpoint mỗi N iteration
TIMING = 1                         # 1: đo tốc độ riêng từng cấu hình trên 1 GPU
TIMING_CONFIGS = "ppo bcrs1"
SMOKE_ONLY = False                 # True: chỉ kiểm tra cài đặt và đo tốc độ, không train

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%%bash
set -e
pip install -q uv
rm -rf /tmp/mjlab
git clone -q --depth 1 -b research/amp-velocity https://github.com/nghiatran0106/mjlab.git /tmp/mjlab
cd /tmp/mjlab
git log --oneline -1
UV_LINK_MODE=copy uv sync --locked --no-dev --extra cu128 > /tmp/uv_sync.log 2>&1 || { tail -20 /tmp/uv_sync.log; exit 1; }
uv run --frozen --no-dev --extra cu128 python -c "import torch, warp as wp; wp.init(); print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(), torch.cuda.get_device_name(0), '| warp CUDA:', wp.is_cuda_available())"

## Kiểm tra nhanh và đo tốc độ (khoảng 5 phút)
Train 10 iteration với 4096 môi trường. Nếu ô này lỗi, dừng lại và gửi log cho Claude.

In [ ]:
%%bash
cd /tmp/mjlab
export MUJOCO_GL=egl
timeout 1500 uv run --frozen --no-dev --extra cu128 train Mjlab-Velocity-Flat-Unitree-G1 \
  --gpu-ids '[0]' --env.scene.num-envs 4096 --agent.max-iterations 10 \
  --agent.logger tensorboard --agent.upload-model False --agent.run-name kaggle-smoke \
  > /tmp/smoke.log 2>&1 || { tail -30 /tmp/smoke.log; exit 1; }
sed 's/\x1b\[[0-9;]*m//g' /tmp/smoke.log | grep -E "Iteration time" | tail -3

In [ ]:
import re, math
log = open('/tmp/smoke.log').read()
times = [float(t) for t in re.findall(r'Iteration time: ([0-9.]+)s', log)]
if times:
    t = sorted(times)[len(times) // 2]
    runs = len(CONFIGS.split()) * len(SEEDS.split())
    per_gpu = math.ceil(runs / 2)  # runs sharing one T4 slow down roughly linearly
    hours = t * per_gpu * ITERS / 3600
    print(f'~{t:.2f} s/iteration, {runs} runs ({per_gpu} per GPU) -> train ≈ {hours:.1f} h + đánh giá ≈ {runs * (ITERS // EVAL_EVERY) * 30 / 3600:.1f} h')
    if hours > 10.5:
        print('CẢNH BÁO: vượt giới hạn 12 giờ của Kaggle — giảm ITERS hoặc số seed.')

## Chạy Giai đoạn 2 (CONFIGS × SEEDS)

In [ ]:
import subprocess
assert not SMOKE_ONLY, 'SMOKE_ONLY=True: dừng ở đây.'
cmd = (f'cd /tmp/mjlab && VARIANT="{VARIANT}" TIMING={TIMING} CONFIGS="{CONFIGS}" SEEDS="{SEEDS}" TIMING_CONFIGS="{TIMING_CONFIGS}" GPUS="0 1" MAX_ITERATIONS={ITERS} EVAL_EVERY={EVAL_EVERY} '
       f'MUJOCO_GL=egl bash src/mjlab/tasks/velocity_amp/scripts/run_phase2.sh 2>&1 '
       f'| tee /kaggle/working/phase2_run.log')
proc = subprocess.run(['bash', '-c', cmd + ' | grep -E "^==|^\\||Done|rror|Traceback"'])
print('exit', proc.returncode)

In [ ]:
%%bash
cp /tmp/mjlab/logs/*_results-*.tgz /kaggle/working/ 2>/dev/null || echo "Không có gói kết quả — xem phase2_run.log"
cp /tmp/mjlab/logs/*_summary.md /kaggle/working/ 2>/dev/null || true
ls -la /kaggle/working/
cat /kaggle/working/*_summary.md 2>/dev/null || true